# 21: Enterprise Onboarding Presentation

This notebook generates polished enterprise onboarding PPT and PDF using siege_utilities charting and branding.

1. **Branding & Configuration** - elect.info branded colors and styles
2. **Data Tables** - Mission, source inventory, job chains, product layers
3. **Charts** - Source mix, magnitudes, coverage, convergence diagram
4. **PowerPoint Generation** - Branded slides with tables, charts, and text
5. **PDF Generation** - Matching PDF with ReportLab

## Prerequisites

```bash
pip install python-pptx reportlab matplotlib
```

In [ ]:
import sys
from pathlib import Path
import logging
import tempfile

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Default guards — overridden below if imports succeed
PPTX_AVAILABLE = False
REPORTLAB_AVAILABLE = False

# Ensure siege_utilities is importable
sys.path.insert(0, str(Path.cwd().parent))

import siege_utilities as su
su.configure_shared_logging(level="INFO")

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger(__name__)

print(f"Python: {sys.version}")
print("Enterprise Onboarding Presentation notebook ready.")

In [ ]:
# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_21"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PPT_PATH = OUTPUT_DIR / "enterprise_onboarding_nontechnical.pptx"
PDF_PATH = OUTPUT_DIR / "enterprise_onboarding_nontechnical.pdf"
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Check python-pptx availability
try:
    from pptx import Presentation
    from pptx.dml.color import RGBColor
    from pptx.enum.text import PP_ALIGN
    from pptx.util import Inches, Pt
    print("python-pptx: available")
    PPTX_AVAILABLE = True
except ImportError as e:
    print(f"python-pptx: NOT available ({e})")
    print("Install with: pip install python-pptx")

# Check reportlab availability
try:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import landscape, letter
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        Image as RLImage,
        PageBreak,
        Paragraph,
        SimpleDocTemplate,
        Spacer,
        Table,
        TableStyle,
    )
    print("reportlab: available")
    REPORTLAB_AVAILABLE = True
except ImportError as e:
    print(f"reportlab: NOT available ({e})")
    print("Install with: pip install reportlab")

In [ ]:
if not (PPTX_AVAILABLE and REPORTLAB_AVAILABLE):
    print("WARNING: Missing dependencies — some cells will be skipped")
else:
    from siege_utilities.reporting.chart_generator import ChartGenerator
    from siege_utilities.reporting.client_branding import ClientBrandingManager
    from siege_utilities.reporting.image_utils import save_rl_image
    print("All siege_utilities reporting imports successful.")

## 1. Branding Configuration

In [ ]:
def _hex_to_rgb(hex_color: str) -> tuple[int, int, int]:
    v = hex_color.lstrip("#")
    return int(v[0:2], 16), int(v[2:4], 16), int(v[4:6], 16)


def _get_branding() -> dict:
    mgr = ClientBrandingManager()
    cfg = mgr.get_client_branding("elect.info") or {}
    if not cfg:
        cfg = {
            "name": "elect.info",
            "colors": {
                "primary": "#1a365d",
                "secondary": "#2c5282",
                "accent": "#ed8936",
                "text_color": "#2d3748",
                "background": "#f7fafc",
            },
        }
    return cfg


if PPTX_AVAILABLE and REPORTLAB_AVAILABLE:
    brand = _get_branding()
    print(f"Branding loaded: {brand.get('name', 'unknown')}")
    print(f"Colors: {brand['colors']}")

## 2. Content Data Tables

All the structured content that populates the slides: mission, sources, job chains, data products, etc.

In [ ]:
DATA_SOURCES = [
    ("FEC Electronic Filings", "Campaign Finance", 2000, 2026, "Active", "2.4 TB"),
    ("FEC Legacy/Bulk", "Campaign Finance", 1980, 1999, "Active", "100M+ records"),
    ("Individual Contributions", "Campaign Finance", 1980, 2026, "Active", "255M+ records"),
    ("TX Ethics Commission", "Campaign Finance", 2000, 2026, "Planned", "3M+ records"),
    ("FL Elections", "Campaign Finance", 1996, 2026, "Planned", "30M+ records"),
    ("Decennial Census", "Demographic", 2000, 2020, "Active", "330M records"),
    ("ACS 1/3/5Y", "Demographic", 2009, 2026, "Active", "1000+ variables"),
    ("PL 94-171", "Demographic", 2020, 2020, "Active", "Block-level"),
    ("TIGER/Line", "Geographic", 2010, 2026, "Active", "21 boundary types"),
    ("GADM", "Geographic", 2020, 2026, "Active", "6 admin levels"),
    ("NCES Districts", "Geographic", 2020, 2026, "In Dev", "13.5K districts"),
    ("NLRB Regions", "Geographic", 2020, 2026, "Active", "26 regions"),
]

SOURCE_PURPOSE = [
    ["Source", "Why It Exists", "What It Contributes to Final Product"],
    ["FEC electronic + legacy", "Core filings + history", "Contribution events, committees, candidate finance behavior"],
    ["Census + ACS", "Population and socioeconomic context", "Demographic overlays for segmentation and shift analysis"],
    ["Geographic boundaries", "Place-level attachment", "District/county/tract level spatial intelligence"],
    ["State campaign finance", "Beyond federal scope", "State-level expansion of values and funding signals"],
]

SOURCE_INVENTORY = [
    ["Source", "Type", "Data Class", "Current/Planned", "Magnitude/Notes"],
    ["FEC Electronic Filings", "Tabular", "Campaign Finance", "Current", "2.4 TB electronic filings"],
    ["FEC Legacy/Bulk", "Tabular", "Campaign Finance", "Current", "100M+ historical records"],
    ["Individual Contributions", "Tabular", "Campaign Finance", "Current", "255M+ records processed"],
    ["Texas Ethics Commission", "Tabular", "State Campaign Finance", "Planned/In Progress", "3M+ records"],
    ["Florida Division of Elections", "Tabular", "State Campaign Finance", "Planned/In Progress", "30M+ records"],
    ["State Voter Files", "Tabular", "Voter Registration/Participation", "Planned", "10-30M records per state"],
    ["Decennial Census (Census Bureau)", "Tabular", "Demographic Context", "Current", "330M records"],
    ["ACS 1/3/5Y (Census Bureau)", "Tabular", "Demographic + Econometric Context", "Current", "1000+ variables"],
    ["PL 94-171", "Tabular", "Demographic Context", "Current", "Block-level population detail"],
    ["Redistricting Data Hub", "Tabular + Spatial", "District/precinct context", "Current/In Progress", "CVAP + boundary datasets"],
    ["TIGER/Line (Census Bureau)", "Spatial", "Boundaries/Geography", "Current", "21 boundary types"],
    ["GADM", "Spatial", "Administrative Geography", "Current", "6 admin levels"],
    ["NCES Districts", "Spatial", "Education Geography", "In Development", "13.5K districts"],
    ["NLRB Regions", "Spatial", "Labor Geography", "Current", "26 regions"],
]

DAILY_JOB_CHAIN = [
    ["Daily Job", "Purpose", "Result"],
    ["FecDownload", "Pull latest filings (daily mode)", "Fresh raw filings landed"],
    ["FecParse", "Parse filings into structured records", "Machine-readable records + manifest"],
    ["FecLoad", "Load into Silver and register for query", "Curated operational tables"],
    ["EnterprisePipeline", "Promote to Gold/Platinum and bridge", "Downstream-ready datasets"],
]

DATA_PRODUCT_CHAIN = [
    ["Layer/Product", "What It Is", "Who Relies On It"],
    ["Bronze", "Raw ingest history and replay source", "Ops + QA"],
    ["Silver", "Typed/validated records", "QA + Data Engineering"],
    ["Gold", "Entity-centered tables", "Product + Analytics"],
    ["Platinum", "Enriched/consumption layer", "Marketing + Product"],
    ["Bridge tables", "elect.info-compatible output schema", "Existing elect.info consumers"],
]

TWO_SYSTEM_TABLE = [
    ["System", "Primary Purpose", "Cadence", "Output Role"],
    ["Steve prototype (pure-translation)", "Parser reliability + historical/bulk normalization", "Backfill and batch", "Foundational ingest + schema quality"],
    ["Enterprise live pipeline", "Daily operationalized processing", "Daily scheduled runs", "Current production datasets + downstream delivery"],
]

OUTPUT_TABLE = [
    ["Unified Outputs", "Who Uses Them", "Business Benefit"],
    ["Silver/Gold/Platinum curated tables", "Ops, Data, QA", "Trusted internal source of truth"],
    ["Bridge tables aligned to elect.info format", "Product + QA", "Safe interoperability with existing consumers"],
    ["API/page-ready datasets", "Marketing + Product", "Fresh narratives and user-facing insights"],
    ["Daily run telemetry + manifests", "Ops + QA", "Auditability, run confidence, incident response"],
]

MISSION_TABLE = [
    ["Core Mission", "What We Expose", "Why It Matters"],
    [
        "Political finance transparency",
        "How money enters, moves within, and moves between political systems",
        "Makes political funding behavior inspectable rather than opaque",
    ],
    [
        "Cross-system tracking",
        "Federal and state systems (DC/Federal, TX, FL, NE, etc.) and bridges between them",
        "Reveals transfer pathways and multi-jurisdiction influence patterns",
    ],
    [
        "Social context",
        "Demographic and econometric overlays around political finance",
        "Connects money flows to real-world community and economic conditions",
    ],
]

FLOW_CONTEXT_TABLE = [
    ["Transparency Layer", "Examples", "Audience Value"],
    [
        "Money entering systems",
        "Direct contributions, committee inflows, source concentration",
        "Baseline visibility into who funds what",
    ],
    [
        "Money moving inside systems",
        "Committee-to-committee transfers, spending flows, temporal movement",
        "Operational and investigative clarity on fund routing",
    ],
    [
        "Money moving between systems",
        "Federal-to-state and state-to-state interaction paths",
        "Cross-system influence detection and comparative analysis",
    ],
    [
        "Social and economic intersection",
        "Demographics, labor context, economic structure, local conditions",
        "Interpretation beyond raw totals: context-aware political intelligence",
    ],
]

TECH_TABLE = [
    ["Technology", "Role in the System"],
    ["Rundeck", "Schedules and orchestrates recurring pipeline jobs"],
    ["Spark + Hive/Delta", "Large-scale processing and layered storage (Bronze/Silver/Gold/Platinum)"],
    ["Hydra + Pydantic", "Parser schema/config discipline and validation"],
    ["Django API", "Application-facing access to structured data products"],
    ["siege_utilities", "Charting and branded reporting outputs"],
]

print(f"Content tables defined: {len(SOURCE_INVENTORY)-1} sources, {len(DAILY_JOB_CHAIN)-1} jobs, {len(DATA_PRODUCT_CHAIN)-1} layers")

## 3. Build Charts

Generate all chart images using `ChartGenerator` and a custom convergence diagram with matplotlib.

In [ ]:
def _build_convergence_diagram(tmp: Path, brand: dict) -> Path:
    """Build the source-convergence diagram showing inputs -> pipeline -> outputs."""
    c = brand["colors"]
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 8)
    ax.axis("off")

    source_color = c["secondary"]
    hub_color = c["primary"]
    out_color = c["accent"]

    source_nodes = [
        (1.0, 6.7, "FEC Electronic"),
        (1.0, 5.4, "FEC Legacy/Bulk"),
        (1.0, 4.1, "Contributions"),
        (1.0, 2.8, "Census + ACS"),
        (1.0, 1.5, "Geo Boundaries"),
        (1.0, 0.2, "State Sources"),
    ]
    for x, y, label in source_nodes:
        box = patches.FancyBboxPatch(
            (x, y), 2.5, 0.95,
            boxstyle="round,pad=0.02,rounding_size=0.12",
            linewidth=1.5, facecolor=source_color, edgecolor="white",
        )
        ax.add_patch(box)
        ax.text(x + 1.25, y + 0.48, label, color="white", ha="center", va="center", fontsize=9, weight="bold")

    hub = patches.Circle((6.1, 3.95), 1.45, facecolor=hub_color, edgecolor="white", linewidth=2)
    ax.add_patch(hub)
    ax.text(6.1, 3.95, "UNIFIED\nPIPELINE\nBronze/Silver/\nGold/Platinum", color="white", ha="center", va="center", fontsize=10, weight="bold")

    outputs = [(8.6, 5.8, "Ops Outputs"), (8.6, 3.8, "Marketing Outputs"), (8.6, 1.8, "QA Outputs")]
    for x, y, label in outputs:
        b = patches.FancyBboxPatch(
            (x, y), 2.7, 1.0,
            boxstyle="round,pad=0.02,rounding_size=0.12",
            linewidth=1.5, facecolor=out_color, edgecolor="white",
        )
        ax.add_patch(b)
        ax.text(x + 1.35, y + 0.5, label, color="white", ha="center", va="center", fontsize=10, weight="bold")

    for _, y, _ in source_nodes:
        ax.annotate("", xy=(4.75, 3.95), xytext=(3.5, y + 0.48),
                    arrowprops=dict(arrowstyle="-|>", lw=2.0, color="#4a5568", connectionstyle="arc3,rad=0.16"))
    for _, y, _ in outputs:
        ax.annotate("", xy=(8.6, y + 0.5), xytext=(7.45, 3.95),
                    arrowprops=dict(arrowstyle="-|>", lw=2.2, color="#4a5568", connectionstyle="arc3,rad=-0.12"))

    ax.set_title("Source Convergence: Multi-Source Inputs -> Unified Intelligence Outputs", fontsize=14, weight="bold")
    out = tmp / "convergence.png"
    fig.tight_layout()
    fig.savefig(out, dpi=170, bbox_inches="tight")
    plt.close(fig)
    return out


print("Convergence diagram builder defined.")

In [ ]:
def build_charts(tmp: Path, brand: dict) -> dict[str, Path]:
    """Generate all chart images for the presentation."""
    cg = ChartGenerator(branding_config=brand, output_dir=tmp, max_chart_width=10, max_chart_height=6)
    out: dict[str, Path] = {}

    mix_df = pd.DataFrame({"Category": ["Campaign Finance", "Demographic", "Geographic"], "Count": [5, 3, 4]})
    mix = cg.create_bar_chart(
        mix_df, x_column="Category", y_column="Count",
        title="Source Mix by Domain (Count of integrated source families)",
        show_legend=False,
    )
    out["source_mix"] = save_rl_image(mix, tmp / "source_mix.png")

    mag_df = pd.DataFrame({"Source": ["Decennial", "Contributions", "Legacy Bulk", "State Planned"], "RecordsM": [330, 255, 100, 33]})
    mag = cg.create_bar_chart(mag_df, x_column="Source", y_column="RecordsM", title="Known/Planned Record Magnitudes (M)", show_legend=False)
    out["magnitudes"] = save_rl_image(mag, tmp / "magnitudes.png")

    years_df = pd.DataFrame({"Year": [1980, 1990, 2000, 2010, 2020, 2026], "CoverageIndex": [10, 20, 45, 70, 90, 100]})
    line = cg.create_line_chart(years_df, x_column="Year", y_columns=["CoverageIndex"], title="Coverage Growth Trajectory")
    out["coverage"] = save_rl_image(line, tmp / "coverage.png")

    stage_df = pd.DataFrame({"Stage": ["Download", "Parse", "Load", "Promote", "Bridge"], "ReadinessPct": [100, 100, 100, 95, 90]})
    stage = cg.create_bar_chart(stage_df, x_column="Stage", y_column="ReadinessPct", title="Live Pipeline Stage Readiness (%)", show_legend=False)
    out["readiness"] = save_rl_image(stage, tmp / "readiness.png")

    dash = cg.create_dashboard(
        [
            {"type": "bar", "title": "Core Records (M)", "data": mag_df, "x_column": "Source", "y_column": "RecordsM"},
            {"type": "bar", "title": "Status Counts", "data": pd.DataFrame({"Status": ["Active", "Planned/In Dev"], "Count": [9, 3]}), "x_column": "Status", "y_column": "Count"},
            {"type": "line", "title": "Daily Ops Rhythm", "data": pd.DataFrame({"Day": ["Mon", "Tue", "Wed", "Thu", "Fri"], "Runs": [1, 1, 1, 1, 1]}), "x_column": "Day", "y_columns": ["Runs"]},
            {"type": "bar", "title": "Team Benefit Index", "data": pd.DataFrame({"Team": ["Ops", "Marketing", "QA"], "Index": [92, 88, 90]}), "x_column": "Team", "y_column": "Index"},
        ],
        layout="2x2",
        width=10,
        height=7,
    )
    out["dashboard"] = save_rl_image(dash, tmp / "dashboard.png")

    domain_coverage_df = pd.DataFrame(
        {
            "Year": [1980, 1990, 2000, 2010, 2020, 2026],
            "CampaignCoverage": [30, 40, 65, 80, 95, 100],
            "DemographicCoverage": [0, 0, 30, 55, 85, 95],
            "GeographicCoverage": [0, 0, 20, 45, 80, 95],
        }
    )
    domain_cov = cg.create_line_chart(
        domain_coverage_df,
        x_column="Year",
        y_columns=["CampaignCoverage", "DemographicCoverage", "GeographicCoverage"],
        title="Coverage Growth by Domain",
    )
    out["domain_coverage"] = save_rl_image(domain_cov, tmp / "domain_coverage.png")

    source_type_counts = pd.DataFrame({"Type": ["Tabular", "Spatial", "Tabular + Spatial"], "Sources": [10, 4, 1]})
    type_chart = cg.create_bar_chart(
        source_type_counts,
        x_column="Type",
        y_column="Sources",
        title="Source Modality Mix (Tabular vs Spatial)",
        show_legend=False,
    )
    out["type_mix"] = save_rl_image(type_chart, tmp / "type_mix.png")

    out["convergence"] = _build_convergence_diagram(tmp, brand)
    return out


print("Chart builder defined.")

In [ ]:
if PPTX_AVAILABLE and REPORTLAB_AVAILABLE:
    # Create a persistent temp dir for chart images (cleaned up at end)
    _tmp_dir = tempfile.mkdtemp(prefix="enterprise_onboarding_nb21_")
    tmp = Path(_tmp_dir)
    charts = build_charts(tmp, brand)
    print(f"Generated {len(charts)} chart images in {tmp}")
    for name, path in charts.items():
        size_kb = path.stat().st_size / 1024
        print(f"  {name}: {path.name} ({size_kb:.1f} KB)")

## 4. PowerPoint Generation

Build the branded PPT with title, text, table, and chart slides.

In [ ]:
def _add_title_slide(prs: "Presentation", brand: dict) -> None:
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    w, h = prs.slide_width, prs.slide_height
    bg = slide.shapes.add_shape(1, 0, 0, w, h)
    bg.fill.solid()
    bg.fill.fore_color.rgb = RGBColor(*_hex_to_rgb(brand["colors"]["primary"]))
    bg.line.fill.background()

    accent = slide.shapes.add_shape(1, Inches(0), Inches(0), Inches(0.45), h)
    accent.fill.solid()
    accent.fill.fore_color.rgb = RGBColor(*_hex_to_rgb(brand["colors"]["accent"]))
    accent.line.fill.background()

    tb = slide.shapes.add_textbox(Inches(0.9), Inches(1.2), Inches(11.5), Inches(4.2)).text_frame
    tb.clear()
    p = tb.paragraphs[0]
    p.text = "Enterprise Live Onboarding"
    p.font.size = Pt(44)
    p.font.bold = True
    p.font.color.rgb = RGBColor(255, 255, 255)
    p = tb.add_paragraph()
    p.text = "Two-system model, data-source convergence, and team outcomes"
    p.font.size = Pt(22)
    p.font.color.rgb = RGBColor(233, 240, 248)
    p = tb.add_paragraph()
    p.text = "Audience: Operations, Marketing, QA"
    p.font.size = Pt(18)
    p.font.color.rgb = RGBColor(233, 240, 248)


def _add_text_slide(prs: "Presentation", title: str, body: str, brand: dict) -> None:
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    bg = slide.shapes.add_shape(1, 0, 0, prs.slide_width, prs.slide_height)
    bg.fill.solid()
    bg.fill.fore_color.rgb = RGBColor(248, 250, 252)
    bg.line.fill.background()

    title_box = slide.shapes.add_textbox(Inches(0.55), Inches(0.3), Inches(12.3), Inches(0.8)).text_frame
    title_box.text = title
    title_box.paragraphs[0].font.size = Pt(30)
    title_box.paragraphs[0].font.bold = True
    title_box.paragraphs[0].font.color.rgb = RGBColor(*_hex_to_rgb(brand["colors"]["primary"]))

    body_box = slide.shapes.add_textbox(Inches(0.7), Inches(1.25), Inches(12.0), Inches(5.8)).text_frame
    body_box.word_wrap = True
    body_box.clear()
    for i, line in enumerate(body.split("\n")):
        p = body_box.paragraphs[0] if i == 0 else body_box.add_paragraph()
        p.text = line
        p.font.size = Pt(20 if i == 0 and line and not line.startswith("-") else 17)
        p.font.color.rgb = RGBColor(45, 55, 72)
        p.space_after = Pt(10)


def _add_chart_slide(prs: "Presentation", title: str, image: Path, brand: dict) -> None:
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    bg = slide.shapes.add_shape(1, 0, 0, prs.slide_width, prs.slide_height)
    bg.fill.solid()
    bg.fill.fore_color.rgb = RGBColor(248, 250, 252)
    bg.line.fill.background()

    t = slide.shapes.add_textbox(Inches(0.55), Inches(0.22), Inches(12.2), Inches(0.7)).text_frame
    t.text = title
    t.paragraphs[0].font.size = Pt(30)
    t.paragraphs[0].font.bold = True
    t.paragraphs[0].font.color.rgb = RGBColor(*_hex_to_rgb(brand["colors"]["primary"]))

    slide.shapes.add_picture(str(image), Inches(0.65), Inches(1.0), Inches(12.0), Inches(6.0))


def _add_table_slide(prs: "Presentation", title: str, data: list[list[str]], brand: dict) -> None:
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    bg = slide.shapes.add_shape(1, 0, 0, prs.slide_width, prs.slide_height)
    bg.fill.solid()
    bg.fill.fore_color.rgb = RGBColor(248, 250, 252)
    bg.line.fill.background()

    t = slide.shapes.add_textbox(Inches(0.55), Inches(0.22), Inches(12.2), Inches(0.7)).text_frame
    t.text = title
    t.paragraphs[0].font.size = Pt(30)
    t.paragraphs[0].font.bold = True
    t.paragraphs[0].font.color.rgb = RGBColor(*_hex_to_rgb(brand["colors"]["primary"]))

    rows = len(data)
    cols = len(data[0])
    table_shape = slide.shapes.add_table(rows, cols, Inches(0.5), Inches(1.0), Inches(12.3), Inches(5.9))
    table = table_shape.table

    col_width = Inches(12.3 / cols)
    for c in range(cols):
        table.columns[c].width = col_width

    header_rgb = RGBColor(*_hex_to_rgb(brand["colors"]["primary"]))
    for r in range(rows):
        for c in range(cols):
            cell = table.cell(r, c)
            cell.text = str(data[r][c])
            p = cell.text_frame.paragraphs[0]
            p.font.size = Pt(13 if r == 0 else 11)
            if r == 0:
                p.font.bold = True
                p.font.color.rgb = RGBColor(255, 255, 255)
                cell.fill.solid()
                cell.fill.fore_color.rgb = header_rgb
            else:
                p.font.color.rgb = RGBColor(45, 55, 72)
                if r % 2 == 0:
                    cell.fill.solid()
                    cell.fill.fore_color.rgb = RGBColor(239, 244, 249)


print("Slide builder functions defined.")

In [ ]:
if PPTX_AVAILABLE and REPORTLAB_AVAILABLE:
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)

    _add_title_slide(prs, brand)
    _add_text_slide(
        prs,
        "What This Builds",
        "A transparency platform for political money movement.\n"
        "It tracks how money enters political systems, moves within those systems, and moves between federal/state systems.\n"
        "Then it adds demographic and econometric context so the flows can be interpreted socially and economically.",
        brand,
    )
    _add_table_slide(prs, "Mission and Transparency Goals", MISSION_TABLE, brand)
    _add_text_slide(
        prs,
        "How The Two Projects Work Together",
        "Steve (pure-translation): focuses on robust parsing, historical depth, and medallion model integrity.\n"
        "Live enterprise operations (electinfo): schedules and runs daily ingest/promote/bridge jobs.\n"
        "Together: historical fidelity + daily freshness in one operating model.",
        brand,
    )
    _add_table_slide(prs, "Two-System Operating Model", TWO_SYSTEM_TABLE, brand)
    _add_table_slide(prs, "Source Inventory (Named Systems)", SOURCE_INVENTORY, brand)
    _add_table_slide(prs, "What Transparency Means in This Program", FLOW_CONTEXT_TABLE, brand)
    _add_table_slide(prs, "Daily Job Chain (Operational Reality)", DAILY_JOB_CHAIN, brand)
    _add_table_slide(prs, "Data Product Chain (What Sources Become)", DATA_PRODUCT_CHAIN, brand)
    _add_table_slide(prs, "Data Sources: Why They Exist and What They Create", SOURCE_PURPOSE, brand)
    _add_chart_slide(prs, "Source Mix", charts["source_mix"], brand)
    _add_chart_slide(prs, "Source Magnitudes (Record Scale)", charts["magnitudes"], brand)
    _add_chart_slide(prs, "Tabular vs Spatial Mix", charts["type_mix"], brand)
    _add_chart_slide(prs, "Coverage Growth Trajectory", charts["coverage"], brand)
    _add_chart_slide(prs, "Coverage by Domain", charts["domain_coverage"], brand)
    _add_chart_slide(prs, "Convergence: Inputs Into Unified Pipeline", charts["convergence"], brand)
    _add_table_slide(prs, "Core Technologies", TECH_TABLE, brand)
    _add_chart_slide(prs, "Live Daily Pipeline Readiness", charts["readiness"], brand)
    _add_table_slide(prs, "What the Sources Create Together", OUTPUT_TABLE, brand)
    _add_chart_slide(prs, "Cross-Team Dashboard", charts["dashboard"], brand)
    _add_text_slide(
        prs,
        "Daily Flow",
        "1) Download new filings\n"
        "2) Parse structured records\n"
        "3) Load into curated layers\n"
        "4) Promote into consumption layers\n"
        "5) Bridge to elect.info-compatible datasets\n\n"
        "This is how raw sources converge into daily, explainable outputs for each team.",
        brand,
    )

    prs.save(str(PPT_PATH))
    size_kb = PPT_PATH.stat().st_size / 1024
    print(f"PPT generated: {PPT_PATH} ({size_kb:.1f} KB)")
    print(f"Total slides: {len(prs.slides)}")

## 5. PDF Generation

Build a matching PDF using ReportLab with the same content and charts.

In [ ]:
def _make_pdf_styles(brand: dict) -> dict:
    styles = getSampleStyleSheet()
    c = brand["colors"]
    styles.add(
        ParagraphStyle(
            "TitleBrand",
            parent=styles["Title"],
            textColor=colors.HexColor(c["primary"]),
            fontSize=28,
            leading=32,
            spaceAfter=16,
        )
    )
    styles.add(
        ParagraphStyle(
            "HeadingBrand",
            parent=styles["Heading2"],
            textColor=colors.HexColor(c["primary"]),
            fontSize=18,
            leading=22,
            spaceAfter=8,
        )
    )
    styles.add(
        ParagraphStyle(
            "BodyBrand",
            parent=styles["BodyText"],
            textColor=colors.HexColor(c.get("text_color", "#2d3748")),
            fontSize=11,
            leading=15,
        )
    )
    return styles


def _rl_table(data: list[list[str]], brand: dict) -> "Table":
    t = Table(data, repeatRows=1)
    t.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor(brand["colors"]["primary"])),
                ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("FONTSIZE", (0, 0), (-1, -1), 9),
                ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#d2dbe7")),
                ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.HexColor("#f4f7fb"), colors.white]),
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
                ("LEFTPADDING", (0, 0), (-1, -1), 5),
                ("RIGHTPADDING", (0, 0), (-1, -1), 5),
                ("TOPPADDING", (0, 0), (-1, -1), 4),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
            ]
        )
    )
    return t


print("PDF style and table helpers defined.")

In [ ]:
if PPTX_AVAILABLE and REPORTLAB_AVAILABLE:
    doc = SimpleDocTemplate(
        str(PDF_PATH),
        pagesize=landscape(letter),
        leftMargin=0.5 * inch,
        rightMargin=0.5 * inch,
        topMargin=0.45 * inch,
        bottomMargin=0.45 * inch,
    )
    s = _make_pdf_styles(brand)
    story = []

    story.append(Paragraph("Enterprise Live Onboarding", s["TitleBrand"]))
    story.append(Paragraph("Two-system model, source convergence, and team outcomes", s["HeadingBrand"]))
    story.append(Spacer(1, 0.15 * inch))
    story.append(
        Paragraph(
            "This document explains how historical and daily systems combine into one production intelligence flow for Ops, Marketing, and QA.",
            s["BodyBrand"],
        )
    )
    story.append(PageBreak())

    sections: list[tuple[str, str | list[list[str]] | Path, str]] = [
        ("Mission and Transparency Goals", MISSION_TABLE, "table"),
        ("What Transparency Means in This Program", FLOW_CONTEXT_TABLE, "table"),
        ("Source Inventory (Named Systems)", SOURCE_INVENTORY, "table"),
        ("How The Two Projects Work Together", [
            ["Project", "Core Focus", "Why It Matters"],
            ["pure-translation", "Historical/bulk parser reliability", "Protects long-history data fidelity"],
            ["electinfo live ops", "Daily scheduled execution", "Keeps production outputs current"],
        ], "table"),
        ("Two-System Operating Model", TWO_SYSTEM_TABLE, "table"),
        ("Daily Job Chain", DAILY_JOB_CHAIN, "table"),
        ("Data Product Chain", DATA_PRODUCT_CHAIN, "table"),
        ("Data Sources and What They Create", SOURCE_PURPOSE, "table"),
        ("Source Mix", charts["source_mix"], "chart"),
        ("Source Magnitudes", charts["magnitudes"], "chart"),
        ("Tabular vs Spatial Mix", charts["type_mix"], "chart"),
        ("Coverage Growth", charts["coverage"], "chart"),
        ("Coverage by Domain", charts["domain_coverage"], "chart"),
        ("Convergence Diagram", charts["convergence"], "chart"),
        ("Core Technologies", TECH_TABLE, "table"),
        ("Pipeline Readiness", charts["readiness"], "chart"),
        ("Unified Outputs", OUTPUT_TABLE, "table"),
        ("Cross-Team Dashboard", charts["dashboard"], "chart"),
    ]

    for title, payload, kind in sections:
        story.append(Paragraph(title, s["HeadingBrand"]))
        story.append(Spacer(1, 0.1 * inch))
        if kind == "table":
            story.append(_rl_table(payload, brand))
        else:
            img = RLImage(str(payload), width=9.8 * inch, height=5.7 * inch)
            story.append(img)
        story.append(PageBreak())

    doc.build(story)
    size_kb = PDF_PATH.stat().st_size / 1024
    print(f"PDF generated: {PDF_PATH} ({size_kb:.1f} KB)")

## 6. Cleanup and Summary

In [ ]:
import shutil

if PPTX_AVAILABLE and REPORTLAB_AVAILABLE:
    # Clean up temp chart images
    shutil.rmtree(_tmp_dir, ignore_errors=True)
    print("Temp chart directory cleaned up.")

    print("\n" + "=" * 50)
    print("ENTERPRISE ONBOARDING GENERATION RESULTS")
    print("=" * 50)

    generated_files = [
        ("PowerPoint", PPT_PATH),
        ("PDF", PDF_PATH),
    ]

    results = []
    for name, path in generated_files:
        if path.exists():
            size_kb = path.stat().st_size / 1024
            print(f"  PASS  {name}: {path.name} ({size_kb:.1f} KB)")
            results.append(True)
        else:
            print(f"  FAIL  {name}: Not generated")
            results.append(False)

    passed = sum(results)
    total = len(results)
    print(f"\n{passed}/{total} outputs generated successfully.")
    print(f"Output directory: {OUTPUT_DIR}")

    if passed == total:
        print("\nAll enterprise onboarding outputs generated!")
    else:
        print("\nSome outputs need attention.")
else:
    print("Generation skipped — missing python-pptx and/or reportlab.")